In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-darkgrid')

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
TICKERS = ["SPY", "QQQ", "IWM", "TLT", "GLD"]

data = {ticker: pd.read_csv(PROCESSED_DIR / f"{ticker}_processed.csv", index_col='Date', parse_dates=True) for ticker in TICKERS}

# We will focus on SPY for this experiment
spy = data['SPY'].copy()

# --- Feature Engineering ---
# Target: Next 5-day return
spy['Target_5d'] = spy['Adj Close'].pct_change(5).shift(-5)

# Features
spy['Mom_5d'] = spy['Adj Close'].pct_change(5)
spy['Mom_10d'] = spy['Adj Close'].pct_change(10)
spy['Mom_20d'] = spy['Adj Close'].pct_change(20)
spy['Vol_20d'] = spy['Daily_Return'].rolling(window=20).std() * np.sqrt(252)

# Volume change (today's volume vs 20-day average)
spy['Volume_Change'] = (spy['Volume'] / spy['Volume'].rolling(window=20).mean()) - 1

# Moving Average relationship (Price / 50-day SMA - 1)
spy['MA_50'] = spy['Adj Close'].rolling(window=50).mean()
spy['MA_Ratio'] = (spy['Adj Close'] / spy['MA_50']) - 1

# Drop NaNs
spy_clean = spy.dropna()

In [2]:
features = ['Mom_5d', 'Mom_10d', 'Mom_20d', 'Vol_20d', 'Volume_Change', 'MA_Ratio']
target = 'Target_5d'

# Train: 2015 to 2021. Test: 2022 to 2024
train = spy_clean.loc[:'2021-12-31']
test = spy_clean.loc['2022-01-01':]

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

# Scale features (StandardScaler is crucial for Ridge and Random Forest)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

Train size: 1713 | Test size: 748


In [3]:
# 1. Baseline: Historical Mean (Naive)
# Predict the average 5-day return from the training set every day
baseline_pred = np.full(len(y_test), y_train.mean())

# 2. Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

# 3. Ridge Regression (L2 Regularization)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
ridge_pred = ridge.predict(X_test_scaled)

# 4. Random Forest (Non-linear)
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)

In [4]:
print("=== OUT-OF-SAMPLE RETURN PREDICTION (2022-2024) ===\n")

models = {
    'Baseline (Mean)': baseline_pred,
    'Linear Regression': lr_pred,
    'Ridge Regression': ridge_pred,
    'Random Forest': rf_pred
}

for name, preds in models.items():
    mse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    # Also calculate the correlation between predictions and actuals (hit rate proxy)
    corr = np.corrcoef(y_test, preds)[0, 1]
    
    print(f"{name}:")
    print(f"  MSE: {mse:.6f}")
    print(f"  R^2: {r2:.4f}")
    print(f"  Correlation: {corr:.4f}\n")

=== OUT-OF-SAMPLE RETURN PREDICTION (2022-2024) ===

Baseline (Mean):
  MSE: 0.000577
  R^2: -0.0013
  Correlation: nan

Linear Regression:
  MSE: 0.000579
  R^2: -0.0048
  Correlation: 0.0480

Ridge Regression:
  MSE: 0.000579
  R^2: -0.0048
  Correlation: 0.0479

Random Forest:
  MSE: 0.000627
  R^2: -0.0880
  Correlation: -0.0429



d:\MARKETLAB\.venv\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\MARKETLAB\.venv\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## Experiment C Conclusion: Momentum & Return Predictability

**1. Naive momentum features fail to predict future returns.**
We constructed features based on historical returns (5-day, 10-day, 20-day), volatility, volume changes, and moving average relationships. We tested these against a naive baseline (predicting the historical mean) using a chronological train/test split (Train: 2015–2021, Test: 2022–2024).

**2. All models failed out-of-sample.**
- **Baseline (Mean):** MSE = 0.000577 (The benchmark to beat)
- **Linear / Ridge Regression:** MSE = 0.000579 (Worse than baseline)
- **Random Forest:** MSE = 0.000627 (Significantly worse)

All models yielded negative R-squared values. This means they were less accurate than simply predicting the average return.

**3. Complexity leads to overfitting.**
The Random Forest, despite being the most complex model, performed the worst. It memorized noise in the training set and failed to generalize to unseen data. Linear and Ridge regression performed slightly better, but the correlation (~0.048) was far too weak to overcome transaction costs.

**Conclusion:** 
There is no evidence that simple momentum features can predict 5-day forward returns in SPY. The null hypothesis (that returns are unpredictable) cannot be rejected. As per our 48-hour scope, we will not force a signal that isn't there, and will move to cross-asset relationships (Experiment D).